# Magnet Design Optimization for Magnetic Resonance Imaging - Linear Accelerator (MRI-LINAC) Radiation Therapy

**Team Members:** Vinicius Jameli, Felix Deng, Jie Mao

**Table of Contents:** 
- [Introduction](#introduction)
- [Problem formulation](#problem-formulation)
- [Baseline model](#baseline-model)
- [Improvement: Sweep line maximal clique constraints](#improvement-sweep-line-maximal-clique-constraints)
- [References](#references)

## Introduction

Modern magnetic resonance imaging (MRI) technology has been one of the most promising modalities in soft-tissue imaging.  Its fine definition of anatomical anomalies has been shown to be beneficial for not only disease diagnostics, but also for coupling with disease treatments such as radiation therapy to increase the precision of treatments [1]. 
However, MRI and other medical technologies are historically designed to function as separate devices, and these systems can often interfere with each other when coupled to function in close proximity. For example, the magnetic field introduced by the charged particles from the X-Ray beams would interfere the magnetic field generated by the MRI machine, disrupting the magnetic field homogeneity and significantly reducing imaging quality [1]. 
Therefore, the evolving needs in modern medicine demands a more flexible, yet general, approach to optimize MRI magnet design when additional design constraints are involved. 

MRI images are generated within a homogeneous magnetic field [2], often created by two coaxial located electromagnets (see Figure 1a) positioned in equidistance from the imaging volume or the volume of interest (VOI), where the magnetic gradients generated by the two magnets are cancelled out under perfect alignment [3]. 
In this project, we aim to extend the existing practices of optimizing magnet design using mixed linear programming to incorporate arbitrarily complex asymmetries [4]. 
More specifically, we are optimizing the spatial location of electromagnetic coils in non-coaxial, yet still concentric, configurations, as shown in Figure 1b. Through this approach, the optimized magnet design will still generate magnetic field remains homogeneous (within a small tolerance), while minimizing the overall weight of the coils to minimize cost. 

<center>
    <img src="./images/coaxial_mangets.jpg" width="80%">
    <p>Figure 1: Schematic representation of coaxial and non-coaxial magnet configurations.</p>
</center>

## Problem Formulation

In order to compute the resulting magnetic field generated by an electromagnetic coil in space, we first define a few variables, as illustrated in Figure 2: 

- Spatial coordinates: $(x, y, z)$
- Internal radius: $r$
- Cross section height: $h$
- Cross section width: $w$
- Number of wire windings: $N$
- Current: $i$

<center>
    <img src="./images/parameters.jpg" width="80%">
    <p>Figure 2. Parameters for magnetic coil and spatial location in space.</p>
</center>

However, modelling the magnetic field generated by an ideal loop of coil based on its geometry involves highly nonlinear relationship. Instead, we exploit the fact that the current $i$ has a linear relationship with the resultant magnetic field. 
Therefore, we only consider $i$ as a variable and fix all other variables, and we are hence solve a large-scale mix-integer programming problem instead of solving a highly nonlinear but small optimization problem. 

In this project, we set $N = 10^7$ and let $w = 1.4 h$. We can further reduce the number of variables by defining $\sigma$ as the area of the cross-section, where $\sigma = h \times w$. Then, we discretize all the spatial coordinates $(x, y, z)$, widths $w$, and internal radius $r$. Such that, we can pre-compute an incidence matrix $\bold{A}$ as an input to the optimization model, where $a_{mn}^s$ is the magnetic field intensity generated by a unit current at the $m$-th target point in imaging domain $D$ generated by the $n$-th coil in the coil domain $\mathcal{N}$ with the $s$-th cross section in a finite set $\mathcal{S}$ that indexes all potential cross-sections that coils may adopt [5], as illustrated in Figure 3. 

<center>
    <img src="./images/domain.jpg" width="75%">
    <p>Figure 3: Illustration of the optimization space in a cross-section of the magnetic coil, reproduced from Ref. [4].</p>
</center>

At the same time, we also make the following assumptions that are commonly used in the field of MRI technology: 

- We assume that the magnetic field is propagated in free space, and the coils are circular with rectangular cross-sections and are coaxial (i.e., share the same $z$-axis). Therefore, it is sufficient to constraint the magnetic field in a semi-circle in the $y=0$ plane due to symmetry, as shown in Figure 3. 
- The resultant magnetic field generated by a coil is linearly proportional to the current in the coil (i.e., linearity), and the resultant magnetic field generated by $N$ coils is given by the superposition of the magnetic fields generated by each of the $N$ coils. This assumption allows for a linear optimization model. 
- It is only necessary to constraint the axial component of the magnetic field ($B_z$) in order to guarantee homogeneity in the VOI, since the radial component ($B_r$ or $B_x$) becomes negligible through quadrature suppression when $B_z$ is homogeneous [3]. Therefore, we only need to constraint $M$ target points surrounding the surface boundary ($\partial D$) of the spherical domain $D$ (i.e., a set of target points $\mathcal{M}$). 
- In reality, the thickness of electrical coils is non-negligible, and we need to avoid overlapping coils in space. Let $\sigma^s$, $w^s$, and $h^s$ denote the area, width, and height of cross-section $s \in \mathcal{S}$. For each pair of candidate coil positions $n$ and $n'$, we define $\mathcal{S}_{nn'}$ as the set of overlapping cross-sections at these two coil locations. Cross-sections $s$ and $s'$ are considered overlapping at coil positions $n$ and $n'$ when $|{z_n - z_{n'}}| < \frac{h^s}{2} + \frac{h^{s'}}{2} + L_z$ and $|{r_n - r_{n'}}| < \frac{w^s}{2} + \frac{w^{s'}}{2} + L_r$, where $L_z$ and $L_r$ are the minimum required gaps between any two coils on the same pole along the $z$- and $r$-axis, respectively. 

Following the formulation originally developed by Dayarian et al. [4], the magnet design optimization problem can essentially be formulated as the following mix-integer linear programming (MILP) problem: 

$$
\begin{align}
\text{minimize} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} r_n \sigma^s x_{n}^{s} \\
\text{subject to} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \leq B_{0} (1 + \varepsilon), \quad \forall m \in \mathcal{M} \\
& \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \geq B_{0} (1 - \varepsilon), \quad \forall m \in \mathcal{M} \\
& |i_{n}^{s}| \leq J_{c} \sigma^{s} x_{n}^{s}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& \sum\limits_{s \in \mathcal{S}} x_{n}^{s} \leq 1, \quad \forall n \in \mathcal{N} \\
& x_{n}^{s} + x_{n'}^{s'} \leq 1, \quad \forall n, n' \in \mathcal{N}, \forall (s, s') \in \mathcal{S}_{nn'} \\
& i_{n}^{s} \text{ free}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& x_{n}^{s} \in \{0,1\}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} 
\end{align}
$$

Note that: 
- $i_n^s$ is a decision variable that takes the current associated with a thick coil at location $n$ in cross-section $s$. $i_n^s$ is used as a free variable in Eqn. 7, because it can take negative values for current passing through the reversed direction. 
- $x_n^s$ is a binary variable that takes the value of 1 if and only if coil $n$ in cross-section $s$ has a non-zero current, and it takes the value of 0 otherwise (Eqn. 8). 
- The presented MILP aims to minimize the total volume of superconductive material required for the magnet construction (Eqn. 1). 
- The constraints in Eqn. 2 and 3 enforce the magnetic field generated at each target point in the VOI is homogeneous with a magnetic intensity of $B_0$, where a small tolerance $\epsilon$ is allowed. 
- The constraint in Eqn. 4 ensures that the current density in active coils remains below the critical current density threshold $J_c$. 
- The constraint in Eqn. 5 ensures that each candidate coil location can take at most one cross-section.
- The constraint in Eqn. 6 prevents coil overlapping by prohibiting imcompatible cross-sections from being simultaneously assigned to any pair of coil positions $n$ and $n'$ according to the predifined set $\mathcal{S}_{nn'}$ (the last assumption discussed above). 

In [3]:
import numpy as np 
from scipy import constants 
from typing import Tuple, Optional 

ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Define constants for the problem 
B_0 = 0.5 # in Tesla, desired magnetic field strength 
EPSILON = 1e-5 # in Tesla, tolerance for magnetic field strength 
J_C = 2.1e8 # in A/m^2, critical current density 

In [ ]:
# Setting the semi-circular imaging domain M on the x-z plane centered at origin 
VOI_RADIUS = 0.2 # in meters 
VOI_NUM_POINTS = 40 

angles = np.linspace(-np.pi / 2, np.pi / 2, VOI_NUM_POINTS) 
voi_boundary_points = np.zeros((VOI_NUM_POINTS, 3)) 
voi_boundary_points[:, 0] = VOI_RADIUS * np.cos(angles) # x-coordinates
voi_boundary_points[:, 1] = 0.0 # y-coordinates (all points lie on the x-z plane)
voi_boundary_points[:, 2] = VOI_RADIUS * np.sin(angles) # z-coordinates

In [ ]:
# Setting default properties of the coil domain N

# Compute the coil domain geometries 
COIL_WIDTH_MIN, COIL_WIDTH_MAX = 0.005, 0.10 # in meters
COIL_NUM_WIDTHS = 20 # number of discrete widths to consider for the coil
CROSS_SECTION_RATIO = 1.4 # w = h * ratio 

coil_widths = np.linspace(COIL_WIDTH_MIN, COIL_WIDTH_MAX, COIL_NUM_WIDTHS)
coil_heights = coil_widths / CROSS_SECTION_RATIO
coil_cross_section_areas = coil_widths * coil_heights

# Compute coil positions in a grid pattern 
COIL_RADIUS_MIN, COIL_RADIUS_MAX = 0.10, 0.48 # in meters
COIL_Z_MIN, COIL_Z_MAX = 0.34, 0.42 # in meters
COIL_GRID_SPACING = 0.02 # in meters
Z_AXIS_X, Z_AXIS_Y = 0.0, 0.0 # x and y coordinates of the z-axis where coils are placed

coil_z_positions = [] 
z = COIL_Z_MIN 
while z <= COIL_Z_MAX + 1e-10: 
    coil_z_positions.append(z) 
    z += COIL_GRID_SPACING
coil_radius_values = [] 
r = COIL_RADIUS_MIN 
while r <= COIL_RADIUS_MAX + 1e-10: 
    coil_radius_values.append(r) 
    r += COIL_GRID_SPACING
    
coil_positions = [] 
coil_radii = []
for z in coil_z_positions: 
    for r in coil_radius_values: 
        coil_positions.append([Z_AXIS_X, Z_AXIS_Y, z]) # all combinations for the north half 
        coil_radii.append(r)
for z in coil_z_positions: 
    for r in coil_radius_values: 
        coil_positions.append([Z_AXIS_X, Z_AXIS_Y, -z]) # all combinations for the south half 
        coil_radii.append(r)
coil_positions = np.array(coil_positions)
coil_radii = np.array(coil_radii)

NUM_COILS = len(coil_positions)

# Other coil parameters
COIL_MIN_DISTANCE = 0.01 # in meters 
COIL_TURNS = 10000000 # number of wire turns 

In [ ]:
from compute_amns import AMNSMatrix
from compute_amns import compute_amns 

In [ ]:
# Since the computation of the AMNS matrix is computationally resource intensive, 
# we directly load a pre-computed matrix from the repository for our experiments.

AMNS_MATRIX = compute_amns(
    opt_points=voi_boundary_points,
    coil_positions=coil_positions,
    coil_radii=coil_radii,
    widths=coil_widths,
    cross_section_ratio=CROSS_SECTION_RATIO,
    N=COIL_TURNS, 
    current=1.0, 
    order_n1=18, order_n2=10, tolerance=1e-4 
)

print("AMNS matrix computed with Bx in shape:", AMNS_MATRIX.Bx.shape)

In [ ]:
# Saving and loading the AMNS matrix using pickle 
def save_amns(amns: AMNSMatrix, filepath: str):
    filepath = str(filepath)
    if filepath.endswith(".npz"):
        np.savez(
            filepath,
            Bx=amns.Bx,
            By=amns.By,
            Bz=amns.Bz,
            num_opt_points=np.array(amns.num_opt_points),
            num_coils=np.array(amns.num_coils),
            num_widths=np.array(amns.num_widths),
        )
    else: 
        raise ValueError("Unsupported file format. Please use .npz for NumPy compressed format.")

def load_amns(filepath: str) -> AMNSMatrix:
    filepath = str(filepath)
    if filepath.endswith(".npz"):
        data = np.load(filepath)
        return AMNSMatrix(
            Bx=data["Bx"],
            By=data["By"],
            Bz=data["Bz"],
            num_opt_points=int(data["num_opt_points"]),
            num_coils=int(data["num_coils"]),
            num_widths=int(data["num_widths"]),
        )
    else: 
        raise ValueError("Unsupported file format. Please use .npz for NumPy compressed format.")


In [ ]:
save_amns(AMNS_MATRIX, "temp_amns.npz")

In [ ]:
AMNS_MATRIX = load_amns("AMNS_semicircle_40.npz")
print("Loaded AMNS matrix with shape:", AMNS_MATRIX.Bx.shape)

In [ ]:
# Compute the S_nn' set to avoid coil overlapping 
# Find all (n1, s1, n2, s2) pairs that would overlap 
OVERLAP_PAIRS = [] 

def _would_overlap(n1: int, s1: int, n2: int, s2: int) -> bool:
    # Z overlap: centers within half-heights + min_distance
    z1 = coil_positions[n1, 2]
    z2 = coil_positions[n2, 2]
    h1 = coil_heights[s1]
    h2 = coil_heights[s2]
    z_overlap = abs(z1 - z2) < (h1 / 2 + h2 / 2 + COIL_MIN_DISTANCE)

    # Radius overlap: radii within half-widths + min_distance
    r1 = coil_radii[n1]
    r2 = coil_radii[n2]
    w1 = coil_widths[s1]
    w2 = coil_widths[s2]
    r_overlap = abs(r1 - r2) < (w1 / 2 + w2 / 2 + COIL_MIN_DISTANCE)

    return z_overlap and r_overlap


# North pole coils 
for n1 in range(NUM_COILS // 2): 
    for n2 in range(n1 + 1, NUM_COILS // 2):
        for s1 in range(COIL_NUM_WIDTHS):
            for s2 in range(COIL_NUM_WIDTHS):
                if _would_overlap(n1, s1, n2, s2):
                    OVERLAP_PAIRS.append((n1, s1, n2, s2))
# South pole coils 
for n1 in range(NUM_COILS // 2, NUM_COILS): 
    for n2 in range(n1 + 1, NUM_COILS):
        for s1 in range(COIL_NUM_WIDTHS):
            for s2 in range(COIL_NUM_WIDTHS):
                if _would_overlap(n1, s1, n2, s2):
                    OVERLAP_PAIRS.append((n1, s1, n2, s2))

## Baseline Model

For the baseline, we implemented the MILP problem modelled with Eqn. 1 to 8 using Gurobi. 

In [ ]:
import gurobipy as gp
from gurobipy import GRB

model_baseline = gp.Model("baseline_coil_optimization") 

# Gurobi solver configurations 
model_baseline.setParam('TimeLimit', 600) # seconds 
model_baseline.setParam('MIPGap', 0.01) # 1% optimality gap
model_baseline.setParam('FeasibilityTol', 1e-9)
model_baseline.setParam('IntFeasTol', 1e-8)
model_baseline.setParam('NumericFocus', 1) # 0=auto, 1=careful, 2=aggressive, 3=very aggressive 
model_baseline.setParam('Threads', 1) 
model_baseline.setParam('OutputFlag', 1) # 0=silent, 1=normal

# Create decision variables 
# i[n, s]: Current through coil n with width s (continuous) [Eqn. 7]
vars_i = model_baseline.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    lb=-GRB.INFINITY,
    ub=GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="i",
)
# x[n, s]: Binary indicator if coil n uses width s [Eqn. 8]
vars_x = model_baseline.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    vtype=GRB.BINARY,
    name="x",
)

# Convert to 2D arrays for easier indexing
array_i = np.array(
    [[vars_i[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)
array_x = np.array(
    [[vars_x[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)

# Set objective: minimize total "coil volume" (radius * area) [Eqn. 1]
# This approximates minimizing total current 
obj = gp.LinExpr(
    (
        coil_radii[:, None] * coil_cross_section_areas[None, :]).flatten().tolist(), 
        list(array_x.flatten()
    )
)
   
model_baseline.setObjective(obj, GRB.MINIMIZE)

In [ ]:
# Add homogeneity constraints on Bz field [Eqn. 2 and 3]
upper_bound = B_0 * (1 + EPSILON)
lower_bound = B_0 * (1 - EPSILON)

if AMNS_MATRIX.Bz.ndim == 3: 
    bz_2d = AMNS_MATRIX.Bz.reshape(VOI_NUM_POINTS, NUM_COILS * COIL_NUM_WIDTHS)
else: 
    bz_2d = AMNS_MATRIX.Bz # already (num_opt_points, num_coils * num_widths)

i_flat = list(array_i.flatten())

for m in range(VOI_NUM_POINTS):
    # Build field expression: sum(AMNS[2,m,n,s] * x[n,s])
    # Compute Bz at point m as sum over coils and widths
    expr = gp.LinExpr(bz_2d[m].tolist(), i_flat)
    
    # Add constraints: lower_bound <= Bz <= upper_bound
    model_baseline.addConstr(expr >= lower_bound, name=f"homo_lower_{m}")
    model_baseline.addConstr(expr <= upper_bound, name=f"homo_upper_{m}")

In [ ]:
# Add symmetry constraint between north and south pole coils 
half_num_coils = NUM_COILS // 2
for n in range(half_num_coils):
    for s in range(COIL_NUM_WIDTHS):
        # North coil n pairs with south coil n + half
        model_baseline.addConstr(
            array_x[n, s] == array_x[n + half_num_coils, s],
            name=f"symmetry_{n}_{s}"
        )

In [ ]:
# Add upper bound constraint on current density [Eqn. 4]
for n in range(NUM_COILS): 
    for s in range(COIL_NUM_WIDTHS): 
        # I_max = J_crit * Area / N
        i_max = J_C * coil_cross_section_areas[s] / COIL_TURNS 
        model_baseline.addConstr(array_i[n, s] <= i_max * array_x[n, s], name=f"current_upper_{n}_{s}")
        model_baseline.addConstr(array_i[n, s] >= -i_max * array_x[n, s], name=f"current_lower_{n}_{s}")

In [ ]:
# Add constraint such that each coil position uses at most one width [Eqn. 5]
for n in range(NUM_COILS): 
    expr = gp.LinExpr() 
    for s in range(COIL_NUM_WIDTHS): 
        expr += array_x[n, s]
    model_baseline.addConstr(expr <= 1, name=f"unique_width_{n}")

In [ ]:
# Add constraint to prevent overlapping coils [Eqn. 6]
for n1, s1, n2, s2 in OVERLAP_PAIRS: 
    # Direct constraint: y[n1, s1] + y[n2, s2] <= 1
    model_baseline.addConstr(
        array_x[n1, s1] + array_x[n2, s2] <= 1, 
        name=f"overlap_{n1}_{s1}_{n2}_{s2}"
    )
    # Cross constraint: y[n1, s2] + y[n2, s1] <= 1
    # Only add if different widths
    if s1 != s2:
        model_baseline.addConstr(
            array_x[n1, s2] + array_x[n2, s1] <= 1,
            name=f"overlap_cross_{n1}_{s2}_{n2}_{s1}",
        )

In [ ]:
# Baseline model in solving the problem 
import time 

model_baseline.update()

start_time = time.time()
model_baseline.optimize() 
solve_time = time.time() - start_time

print("Solve time (seconds):", solve_time)
print("Optimization status:", model_baseline.Status)
print("Objective value:", model_baseline.ObjVal if model_baseline.SolCount > 0 else float("inf"))

## Improvement: Sweep Line Maximal Clique Constraints 

To improve the baseline model, we focus on tackling the pairwise overlapping constraint, i.e. Eq (6) in the Baseline Formulation.

$$
x_{n}^{s} + x_{n'}^{s'} \leq 1, \quad \forall n, n' \in \mathcal{N}, \forall (s, s') \in \mathcal{S}_{nn'}
$$

This approach requires $\mathcal{O}((N \cdot S)^2)$ constraints in the worst case, which is prohibitively large - creating a massive memory bottleneck and weakening the linear programming (LP) relaxation of the problem.

The first step towards strengthening this formulation is defining its graph representation. Let $G=(\mathcal{V},\mathcal{E})$ be a **conflict graph** where each vertex $v \in \mathcal{V} = \mathcal{N} \times \mathcal{S} $ represents coil $(n,s)$, and an edge $e \in \mathcal{E}$ exists between two vertices if the corresponding coils overlap.
A **clique** $\mathcal{C} \subseteq V$ is a subset of vertices where every two distinct vertices are adjacent.
Consequently, at most one coil from any clique can be active simultaneously.
In the literature, whenever set packing constraints arise, the practice has been to create the conflict graph of these overlaps to extract cliques and tighten the formulation \cite{rodrigues2017clique} - in particular, **maximal cliques**. A maximal clique is a clique that can not be extended by including one more adjacent vertex, i.e., it is not a subset of a larger clique. Therefore, a set-packing constraint from a clique $\mathcal{C}$ is simply defined as:

$$\sum_{(n,s) \in \mathcal{C}} x_n^s \leq 1 \tag{6a}$$

This constraint is significantly stronger than the pairwise formulation, as it cuts off the fractional solutions
(e.g., $x_i=x_j=x_k=0.5$) that satisfy pairwise constraints but violate the clique constraint (e.g., $x_i+x_j+x_k \le 1$).

We replace Eq (6) with the Eq (6a) above, giving us the alternative formulation.

$$
\begin{align}
\text{minimize} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} r_n \sigma^s x_{n}^{s} \\
\text{subject to} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \leq B_{0} (1 + \varepsilon), \quad \forall m \in \mathcal{M} \\
& \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \geq B_{0} (1 - \varepsilon), \quad \forall m \in \mathcal{M} \\
& |i_{n}^{s}| \leq J_{c} \sigma^{s} x_{n}^{s}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& \sum\limits_{s \in \mathcal{S}} x_{n}^{s} \leq 1, \quad \forall n \in \mathcal{N} \\
& \sum_{(n,s) \in \mathcal{C}} x_n^s \leq 1 \tag{6a} \\
& i_{n}^{s} \text{ free}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \tag{7} \\
& x_{n}^{s} \in \{0,1\}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \tag{8}
\end{align}
$$



### Sweepline Maximal Clique Algorithm

Since all the coils share the same axial axis (i.e., the $z$ axis), the coil overlap boils down to overlap of rectangles in the plane $y=0$. Crucially, a family of subsets of the set of rectangles $R_i$ $(i = 1,\dots,n)$ with sides parallel to the axes satisfies the Helly property. That is, for any subset of $(R_{i_1},\ldots,R_{i_k})$ of rectangles of which every pair intersects, all the rectangles contained in the subset have nonempty intersection \cite{imai1983finding}. In other words: two (or more) rectangles overlap if there is a region in space shared by the three rectangles. Luckily,  \cite{imai1983finding} have showed that there is a polynomial algorithm, $\mathcal{O}(n\log n)$ for finding the maximum clique in an intersection graph of rectangles, and their algorithm can be easily modified to enumerate all the maximal cliques in polynomial time just looking at the geometry of overlap, without ever needing to build the conflict graph.
Imai \& Asano's algorithm falls into the category of sweep line algorithms, commonly used for solving planar problems. In our case, we sweep a line over the plane $y=0$ checking for "events" - in our case, rectangle overlaps at a certain point in space. Every time we hit an event, we compute the clique that happened at that point - that is, we enumerate the rectangles overlapping at that point - and move to the next event. 

Thus, our \textbf{methodological contribution} is a new way of modeling set-packing constraints that represents non-overlapping of rectangles in 2D. Our method exploits of the geometry to derive all maximal cliques in polynomial time and therefore derive a very tight formulation for the problem with orders of magnitude less constraints. This shows that instead of constructing the conflict graph, much insight can be extract directly from the geometry of overlap! 

<center>
    <img src="./images/Algorithm_1.png" width="100%">
</center>

Below is an implementation of the alternative formulation using the Sweep-line Maximal Clique Enumeration algorithm.

In [ ]:
# ---------------------------------------------------------
# SWEEP-LINE MAXIMAL CLIQUE ALGORITHM
# ---------------------------------------------------------

import time
from dataclasses import dataclass, field
from typing import FrozenSet, List, Optional, Set, Tuple
import numpy as np

# Type alias for a variable index pair
VarIdx = Tuple[int, int]  # (coil_idx, width_idx)
Clique = FrozenSet[VarIdx]

@dataclass
class SweepLineStats:
    num_raw_cliques: int
    num_maximal_cliques: int
    total_members: int
    elapsed_sweep_s: float
    elapsed_filter_s: float
    elapsed_total_s: float

_END = 0    
_START = 1

@dataclass
class SweepLineCliqueEnumerator:
    positions: np.ndarray   # (num_coils, 3)
    radii: np.ndarray       # (num_coils,)
    widths: np.ndarray      # (num_widths,)
    heights: np.ndarray     # (num_widths,)
    min_distance: float

    last_run_stats: Optional[SweepLineStats] = field(default=None, init=False, repr=False)

    def enumerate_cliques(self, north_only: bool = True) -> List[Clique]:
        t_start = time.perf_counter()

        num_coils = len(self.positions)
        num_widths = len(self.widths)
        L = self.min_distance

        # Determine which coil indices to consider
        if north_only:
            coil_range = range(num_coils // 2)
        else:
            coil_range = range(num_coils)

        # Precompute effective intervals
        half_L = 0.5 * L
        z_lo: dict[VarIdx, float] = {}
        z_hi: dict[VarIdx, float] = {}
        r_lo: dict[VarIdx, float] = {}
        r_hi: dict[VarIdx, float] = {}

        for n in coil_range:
            z_n = self.positions[n, 2]
            r_n = self.radii[n]
            for s in range(num_widths):
                h_s = self.heights[s]
                w_s = self.widths[s]
                idx: VarIdx = (n, s)
                z_lo[idx] = z_n - h_s * 0.5 - half_L
                z_hi[idx] = z_n + h_s * 0.5 + half_L
                r_lo[idx] = r_n - w_s * 0.5 - half_L
                r_hi[idx] = r_n + w_s * 0.5 + half_L

        all_pairs: List[VarIdx] = list(z_lo.keys())

        # Build z-events
        z_events: List[Tuple[float, int, VarIdx]] = []
        for idx in all_pairs:
            z_events.append((z_lo[idx], _START, idx))
            z_events.append((z_hi[idx], _END, idx))

        # Sort: primary key = z, secondary = event_type (END=0 < START=1)
        z_events.sort(key=lambda e: (e[0], e[1]))

        # Plane sweep (outer, z-axis)
        active: Set[VarIdx] = set()
        raw_cliques: Set[Clique] = set() 

        for i, (z_coord, etype, idx) in enumerate(z_events):
            if etype == _START:
                active.add(idx)
            else:  # _END
                active.discard(idx)

            # Emit 1D cliques when there is a gap to the next event
            if active and i + 1 < len(z_events) and z_events[i + 1][0] > z_coord:
                for clique in self._solve_1d_cliques(active, r_lo, r_hi):
                    raw_cliques.add(clique)

        t_sweep = time.perf_counter()
        elapsed_sweep = t_sweep - t_start

        # Filter: remove proper subsets
        maximal = self._filter_subsets(raw_cliques)

        elapsed = time.perf_counter() - t_start
        elapsed_filter = elapsed - elapsed_sweep
        total_members = sum(len(c) for c in maximal)

        self.last_run_stats = SweepLineStats(
            num_raw_cliques=len(raw_cliques),
            num_maximal_cliques=len(maximal),
            total_members=total_members,
            elapsed_sweep_s=elapsed_sweep,
            elapsed_filter_s=elapsed_filter,
            elapsed_total_s=elapsed,
        )

        print(
            f"[SweepLine] Elapsed: {elapsed:.4f}s "
            f"(sweep: {elapsed_sweep:.4f}s, filter: {elapsed_filter:.4f}s) | "
            f"Raw cliques: {len(raw_cliques)} | "
            f"Maximal cliques: {len(maximal)}"
        )

        return maximal

    def _solve_1d_cliques(self, active: Set[VarIdx], r_lo: dict, r_hi: dict) -> List[Clique]:
        r_events: List[Tuple[float, int, VarIdx]] = []
        for idx in active:
            r_events.append((r_lo[idx], _START, idx))
            r_events.append((r_hi[idx], _END, idx))

        r_events.sort(key=lambda e: (e[0], e[1]))

        candidates: List[Clique] = []
        curr: Set[VarIdx] = set()

        for r_coord, etype, idx in r_events:
            if etype == _START:
                curr.add(idx)
            else:  # _END
                if len(curr) >= 2:
                    candidates.append(frozenset(curr))
                curr.discard(idx)

        return candidates

    @staticmethod
    def _filter_subsets(cliques: Set[Clique]) -> List[Clique]:
        sorted_cliques = sorted(cliques, key=len, reverse=True)
        maximal: List[Clique] = []
        index: dict = {}  

        for c in sorted_cliques:
            pivot: Optional[list] = None
            min_len = float("inf")
            new_element_found = False

            for elem in c:
                elem_list = index.get(elem)
                if elem_list is None:
                    new_element_found = True
                    break
                if len(elem_list) < min_len:
                    pivot = elem_list
                    min_len = len(elem_list)

            if new_element_found:
                maximal.append(c)
                for elem in c:
                    index.setdefault(elem, []).append(c)
                continue

            is_subset = pivot is not None and any(c < sup for sup in pivot)

            if not is_subset:
                maximal.append(c)
                for elem in c:
                    index.setdefault(elem, []).append(c)

        return maximal

In [ ]:
# Initialize the enumerator with the geometric variables defined in baseline
clique_enumerator = SweepLineCliqueEnumerator(
    positions=coil_positions,
    radii=coil_radii,
    widths=coil_widths,
    heights=coil_heights,
    min_distance=COIL_MIN_DISTANCE
)

# Run the algorithm to get the list of maximal cliques for the north pole
cliques_list = clique_enumerator.enumerate_cliques(north_only=True)

In [ ]:
# ---------------------------------------------------------
# INITIALIZE MODEL: Sweepline Maximal Clique
# ---------------------------------------------------------
model_alt = gp.Model("alternative_coil_optimization") 

# Gurobi solver configurations (Matching your baseline)
model_alt.setParam('TimeLimit', 600) 
model_alt.setParam('MIPGap', 0.01) 
model_alt.setParam('FeasibilityTol', 1e-9)
model_alt.setParam('IntFeasTol', 1e-8)
model_alt.setParam('NumericFocus', 1) 
model_alt.setParam('Threads', 1) 
model_alt.setParam('OutputFlag', 1) 

# Create decision variables 
vars_i_alt = model_alt.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    lb=-GRB.INFINITY,
    ub=GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name="i_alt",
)

vars_x_alt = model_alt.addVars(
    NUM_COILS,
    COIL_NUM_WIDTHS,
    vtype=GRB.BINARY,
    name="x_alt",
)

# Convert to 2D arrays
array_i_alt = np.array(
    [[vars_i_alt[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)
array_x_alt = np.array(
    [[vars_x_alt[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(NUM_COILS)],
    dtype=object,
)

# Set objective: minimize total "coil volume" (radius * area) [Eqn. 1]
obj_alt = gp.LinExpr(
    (coil_radii[:, None] * coil_cross_section_areas[None, :]).flatten().tolist(), 
    list(array_x_alt.flatten())
)
model_alt.setObjective(obj_alt, GRB.MINIMIZE)

In [ ]:
# Add homogeneity constraints on Bz field [Eqn. 2 and 3]
upper_bound = B_0 * (1 + EPSILON)
lower_bound = B_0 * (1 - EPSILON)

if AMNS_MATRIX.Bz.ndim == 3: 
    bz_2d = AMNS_MATRIX.Bz.reshape(VOI_NUM_POINTS, NUM_COILS * COIL_NUM_WIDTHS)
else: 
    bz_2d = AMNS_MATRIX.Bz # already (num_opt_points, num_coils * num_widths)

i_flat_alt = list(array_i_alt.flatten())

for m in range(VOI_NUM_POINTS):
    # Build field expression: sum(AMNS[2,m,n,s] * x[n,s])
    expr = gp.LinExpr(bz_2d[m].tolist(), i_flat_alt)
    
    # Add constraints: lower_bound <= Bz <= upper_bound
    model_alt.addConstr(expr >= lower_bound, name=f"homo_lower_{m}")
    model_alt.addConstr(expr <= upper_bound, name=f"homo_upper_{m}")

In [ ]:
# Add symmetry constraint between north and south pole coils 
half_num_coils = NUM_COILS // 2
for n in range(half_num_coils):
    for s in range(COIL_NUM_WIDTHS):
        # North coil n pairs with south coil n + half
        model_alt.addConstr(
            array_x_alt[n, s] == array_x_alt[n + half_num_coils, s],
            name=f"symmetry_{n}_{s}"
        )

In [ ]:
# Add upper and lower bound constraints on current density [Eqn. 4]
for n in range(NUM_COILS): 
    for s in range(COIL_NUM_WIDTHS): 
        # I_max = J_crit * Area / N
        i_max = J_C * coil_cross_section_areas[s] / COIL_TURNS 
        model_alt.addConstr(array_i_alt[n, s] <= i_max * array_x_alt[n, s], name=f"current_upper_{n}_{s}")
        model_alt.addConstr(array_i_alt[n, s] >= -i_max * array_x_alt[n, s], name=f"current_lower_{n}_{s}")

In [ ]:
# Apply the sweep-line maximal clique constraints
t0 = time.time()
n_cliques = len(cliques_list)
print(f"Starting to add {n_cliques} clique constraints...")

# Note: We are iterating through the list generated in step 1.
# If you imported your teammate's class directly, you can also just call its .generate() method.
for i, clique in enumerate(cliques_list):
    if i % 10000 == 0 and i > 0:
        print(f"  Added {i}/{n_cliques} constraints...")
        
    expr = gp.LinExpr()
    # Each clique is a frozenset of (n, s) tuples
    for n, s in clique:
        expr += array_x_alt[n, s]
        
    model_alt.addConstr(expr <= 1, name=f"clique_{i}")

print(f"Finished adding clique constraints in {time.time() - t0:.2f} seconds.")

In [ ]:
# Alternative model in solving the problem
model_alt.update()

start_time = time.time()
model_alt.optimize() 
solve_time = time.time() - start_time

print("Solve time (seconds):", solve_time)
print("Optimization status:", model_alt.Status)
print("Objective value:", model_alt.ObjVal if model_alt.SolCount > 0 else float("inf"))

Concepts yet to cover:
- Cycle of odd length without chords
- Sweep Line algorithm: Run a line in one direction, track rectangles 'started' but not 'ended' at the same time. Compute clique when ractangle overlaps at a certain point
- Edge Clique Cover: A set of cliques that covers all edges (possible conflicts)
- Edge Clique Cover Number: Minimum # of sets in this cover
- Discussions/propositions

## References

1. G. P. Liney, B. Whelan, B. Oborn, M. Barton, and P. Keall, “MRI-Linear Accelerator Radiotherapy Systems,” Clinical Oncology, vol. 30, no. 11, pp. 686–691, Nov. 2018, doi: 10.1016/j.clon.2018.08.003.
1. A. Sattarov, P. McIntyre, and L. Motowidlo, “High-Field Open MRI for Breast Cancer Screening,” IEEE Trans. Appl. Supercond., vol. 25, no. 3, pp. 1–5, Jun. 2015, doi: 10.1109/TASC.2014.2377049.
1. Hao Xu, S. M. Conolly, G. C. Scott, and A. Macovski, “Homogeneous magnet design using linear programming,” IEEE Trans. Magn., vol. 36, no. 2, pp. 476–483, Mar. 2000, doi: 10.1109/20.825817.
1. I. Dayarian, T. C. Y. Chan, D. Jaffray, and T. Stanescu, “A mixed-integer optimization approach for homogeneous magnet design,” Technology, vol. 06, no. 02, pp. 49–58, Jun. 2018, doi: 10.1142/S2339547818500036.
1. L. K. Forbes, S. Crozier, and D. M. Doddrell, “Rapid computation of static fields produced by thick circular solenoids,” IEEE Trans. Magn., vol. 33, no. 5, pp. 4405–4410, Sep. 1997, doi: 10.1109/20.620453.